# Final Project
#### Team: Arjun Narasimhan & Leon Katarzynski

## Design (04/08/2026)

## 1. Problem and Content

I think we all still experience the same thing: Going to the Science Center, searching for a room, the post office or where again is HUIT? The Science Center is an enormous building, hosting shops, classrooms, labs, in between administrative office and somewhere there, whether in the 8th floor or in the basement, a classroom that you need to find at some point (I did actually come late to my classes because I did not know that there is a classroom in the underground floor!)
However, Google Maps stops at the front door, yet that is where the building becomes confusing. 

Our Final Project solves this with an interactive indoor map. When a user enters the building, the app detects the transition via GPS geofence and switches from outdoor Google Maps routing to a searchable floor-plan view. The user can search for any room, office, or amenity (e.g. *"vending machine"*, *"Room 301"*) and the app highlights a walking path to that destination across floors.

This notebook documents the first steps in the the search and path-finding logic, implemented in Python as a prototype before it is ported to the JavaScript frontend. While we requested access to the official floor plans of the Science Center, we did not receive any response yet. Therefore, the following script works with a default building, roughly sketched to illustrate the method of our code. 

## 2. Approach and Content

The problem decomposes into two sub-problems:

1. **Search** — the user inputs the room, amenity etc. into the search bar.
2. **Path-finding** — given the user's input, the code returns the fastest way to arrive the desired destination.

## 3. Data Structures

The key insight: **the building is a graph.** Rooms are nodes. Two rooms share an edge if a student can walk directly from one to the other without passing through a third room. Stairwells are special nodes whose edges cross between floors — this is how the algorithm knows to go *up* to reach a room on floor 2 from floor 1.

## 4. Algorithm: Breadth-First Search

Finding the shortest path between two rooms is an instance of **shortest path on an unweighted graph**, which is solved optimally by **BFS**. BFS explores nodes layer by layer: first all rooms 1 hop away, then 2 hops away, and so on. Because every step has equal cost, the first time BFS reaches the target it has found the path with the fewest hops.

## 5. Demo Code

### 5.1 Default building


As pointed out in the introduction, we yet do not have access to the Science Center Floor Plan data. Therefore, the following code sketches a default building with three floors in the JSON schema we hope to use for the actual Science Center map.

In [1]:
# Schema overview:
# - Each room has a unique "id", a  "name", a "type", floor number,
#   pixel coordinates within the picture/map, a "neighbors" list of directly connected
#   room ids (meaning you can enter one room through the other), and an optional "description"
#   (f.e. containig whether it is a class room, has any unique features, etc.).
# - Stairwells on adjacent floors count each other as neighbors, allowing BFS to identify them
#   as identical (different floor but same stairwell)
# - Amenity rooms use "amenitySubtype" (e.g., "vending", "restroom").
#
# Floor layout::
#
#   FLOOR 1: sc-101, sc-102, sc-103, sc-rest-1, sc-stair-1
#   FLOOR 2: sc-stair-1, sc-stair-2
#            sc-stair-2, sc-201, sc-202
#   FLOOR 3: sc-stair-2, sc-stair-3
#            sc-stair-3, sc-301, sc-302, sc-vend-1
#
# pixelCoords give each room an (x, y) position on a coordinate system of the floor plan
# Stairwells stack at the same (x, y) across floors, reflecting real buildings.

SAMPLE_BUILDING: dict = {
    "buildingId": "science-center",
    "name": "Harvard Science Center (Sample)",
    "location": {"lat": 42.37652, "lng": -71.11639},
    "floors": [
        {
            "floorNumber": 1,
            "label": "Ground Floor",
            "rooms": [
                {
                    "id": "sc-101",
                    "name": "Lecture Hall A",
                    "type": "lecture",
                    "floor": 1,
                    "pixelCoords": {"x": 100, "y": 120},
                    "neighbors": ["sc-102"],
                    "description": "Large lecture hall, seats 200",
                },
                {
                    "id": "sc-102",
                    "name": "Classroom B",
                    "type": "classroom",
                    "floor": 1,
                    "pixelCoords": {"x": 250, "y": 120},
                    "neighbors": ["sc-101", "sc-103"],
                    "description": "Mid-size classroom with whiteboards",
                },
                {
                    "id": "sc-103",
                    "name": "Lobby",
                    "type": "lobby",
                    "floor": 1,
                    "pixelCoords": {"x": 400, "y": 280},
                    "neighbors": ["sc-102", "sc-rest-1"],
                    "description": "Main ground-floor lobby",
                },
                {
                    "id": "sc-rest-1",
                    "name": "Restroom 1",
                    "type": "amenity",
                    "amenitySubtype": "restroom",
                    "floor": 1,
                    "pixelCoords": {"x": 550, "y": 120},
                    "neighbors": ["sc-103", "sc-stair-1"],
                    "description": "Ground floor restroom",
                },
                {
                    "id": "sc-stair-1",
                    "name": "Main Stairwell (Floor 1)",
                    "type": "stairwell",
                    "floor": 1,
                    "pixelCoords": {"x": 700, "y": 280},
                    # sc-stair-2 is the same stairwell one floor up —
                    # this edge crosses floors in our graph.
                    "neighbors": ["sc-rest-1", "sc-stair-2"],
                    "description": "Main stairwell, connects floors 1 and 2",
                },
            ],
        },
        {
            "floorNumber": 2,
            "label": "Second Floor",
            "rooms": [
                {
                    "id": "sc-stair-2",
                    "name": "Main Stairwell (Floor 2)",
                    "type": "stairwell",
                    "floor": 2,
                    "pixelCoords": {"x": 700, "y": 280},
                    # Bridges floor 1 below and floor 3 above.
                    "neighbors": ["sc-stair-1", "sc-201", "sc-stair-3"],
                    "description": "Main stairwell, connects floors 1-3",
                },
                {
                    "id": "sc-201",
                    "name": "Research Office",
                    "type": "office",
                    "floor": 2,
                    "pixelCoords": {"x": 550, "y": 120},
                    "neighbors": ["sc-stair-2", "sc-202"],
                    "description": "Faculty research office",
                },
                {
                    "id": "sc-202",
                    "name": "Computer Lab",
                    "type": "lab",
                    "floor": 2,
                    "pixelCoords": {"x": 400, "y": 120},
                    "neighbors": ["sc-201"],
                    "description": "General-purpose computer lab",
                },
            ],
        },
        {
            "floorNumber": 3,
            "label": "Third Floor",
            "rooms": [
                {
                    "id": "sc-stair-3",
                    "name": "Main Stairwell (Floor 3)",
                    "type": "stairwell",
                    "floor": 3,
                    "pixelCoords": {"x": 700, "y": 280},
                    "neighbors": ["sc-stair-2", "sc-301"],
                    "description": "Main stairwell, top landing",
                },
                {
                    "id": "sc-301",
                    "name": "Study Room 3A",
                    "type": "study",
                    "floor": 3,
                    "pixelCoords": {"x": 550, "y": 120},
                    "neighbors": ["sc-stair-3", "sc-302"],
                    "description": "Quiet study room on the top floor",
                },
                {
                    "id": "sc-302",
                    "name": "Seminar Room",
                    "type": "classroom",
                    "floor": 3,
                    "pixelCoords": {"x": 400, "y": 120},
                    "neighbors": ["sc-301", "sc-vend-1"],
                    "description": "Small seminar room, seats 20",
                },
                {
                    "id": "sc-vend-1",
                    "name": "Vending Machine Alcove",
                    "type": "amenity",
                    "amenitySubtype": "vending",
                    "floor": 3,
                    "pixelCoords": {"x": 250, "y": 280},
                    "neighbors": ["sc-302"],
                    "description": "Snack and drink vending machines",
                },
            ],
        },
    ],
}

DEFAULT_BUILDING = SAMPLE_BUILDING

print(f"Loaded building: {SAMPLE_BUILDING['name']}")
print(f"Floors: {len(SAMPLE_BUILDING['floors'])}")
print(f"Total rooms: {sum(len(f['rooms']) for f in SAMPLE_BUILDING['floors'])}")


Loaded building: Harvard Science Center (Sample)
Floors: 3
Total rooms: 12


### 5.2 Indexing: Hash Maps for O(1) Lookup

Flatten all floors into two lookup dicts: one keyed by room id (for direct access during BFS to navigate the user to get there), one sorted by (sub)type to search for amenities, rooms, lecture halls specifically.

In [ ]:
def build_room_index(building: dict) -> dict:
    """
    Flatten all rooms across all floors into a single dict keyed by room id.

    Ensures the 'floor' field on each room matches the floor it was declared
    under (defensive — the sample data is consistent, but future imports
    from real data files may not be).

    Args:
        building: The full building dict following the SAMPLE_BUILDING schema.

    Returns:
        A dict mapping room_id (str) -> room object (dict).
        Example: {"sc-101": {"id": "sc-101", "name": "Lecture Hall A", ...}, ...}
    """
    index: dict = {}

    for floor in building["floors"]:
        floor_number: int = floor["floorNumber"]
        for room in floor["rooms"]:
            # Shallow-copy so mutations here don't modify the original.
            room_copy = dict(room)
            # Overwrite 'floor' with the authoritative value from the
            # enclosing floor object in case of a typo in the room record.
            room_copy["floor"] = floor_number
            index[room_copy["id"]] = room_copy

    return index


def build_type_index(building: dict) -> dict:
    """
    Build a secondary index grouping rooms by their queryable type string.

    For most rooms the key is the 'type' field (e.g., "lecture", "office").
    For amenity rooms that carry an 'amenitySubtype' (e.g., "vending",
    "restroom"), the subtype is used as the key instead of the generic
    "amenity" string, so a user search for "vending" works directly.

    Args:
        building: The full building dict.

    Returns:
        A dict mapping type_string (str) -> list of room objects.
        Example: {"lecture": [...], "vending": [...], "restroom": [...]}
    """
    type_index: dict = {}

    for floor in building["floors"]:
        for room in floor["rooms"]:
            # Use amenitySubtype as the key when present (fine-grained lookup).
            if room.get("amenitySubtype"):
                key: str = room["amenitySubtype"]
            else:
                key = room["type"]

            # setdefault initializes the list on first encounter, then appends.
            type_index.setdefault(key, []).append(room)

    return type_index


# --- Build the indexes from our sample building ---
room_index = build_room_index(SAMPLE_BUILDING)
type_index = build_type_index(SAMPLE_BUILDING)

print(f"Total rooms indexed: {len(room_index)}")
print(f"Type keys: {sorted(type_index.keys())}")
print(f"Rooms under 'vending': {[r['id'] for r in type_index.get('vending', [])]}")


Total rooms indexed: 12
Type keys: ['classroom', 'lab', 'lecture', 'lobby', 'office', 'restroom', 'stairwell', 'study', 'vending']
Rooms under 'vending': ['sc-vend-1']


### 5.3 Search: Substring and Type-Indexed Matching

Case-insensitive substring search over every room's text fields, plus a fast path that checks the type index for exact category keywords. Deduplicates results and sorts them by floor.

In [ ]:
def search_rooms(query: str, room_index: dict, type_index: dict) -> list:
    """
    Search for rooms matching a case-insensitive query string.

    Searches the following fields on each room:
        - 'id'
        - 'name'
        - 'type'
        - 'amenitySubtype' (if present)
        - 'description' (if present)

    Also performs an exact match against the type_index keys, so that
    a query like "vending" reliably returns amenity rooms whose 'type'
    field says "amenity" (as long as their amenitySubtype matches).

    Results are deduplicated (a room matched by multiple fields appears
    once), then sorted by floor number and id for deterministic output.

    Args:
        query:      The search string entered by the user.
        room_index: {room_id: room_object} from build_room_index().
        type_index: {type_string: [rooms]} from build_type_index().

    Returns:
        A list of matching room dicts, sorted by (floor, id).
    """
    q: str = query.strip().lower()   # Normalize once; reuse for every check.
    matched_ids: set = set()         # Set deduplicates matches across paths.

    # substring match over every room's text fields
    for room_id, room in room_index.items():
        # .get() handles not-needed fields like description.
        fields_to_search = [
            room.get("id", ""),
            room.get("name", ""),
            room.get("type", ""),
            room.get("amenitySubtype", ""),
            room.get("description", ""),
        ]
        # shortcut as soon as any one field matches.
        for field in fields_to_search:
            if q in field.lower():
                matched_ids.add(room_id)
                break

    # exact type_index key match
    # case where query == "vending" but room.type == "amenity"
    if q in type_index:
        for room in type_index[q]:
            matched_ids.add(room["id"])  # set handles duplicates.

    results: list = [room_index[rid] for rid in matched_ids]
    results.sort(key=lambda r: (r["floor"], r["id"]))

    return results


### 5.4 Path Finding: BFS on the Room Graph

The standard implementation of BFS  uses `collections.deque`. The graph is built implicitly from each room's `neighbors` list. Stairwells on adjacent floors reference each other as neighbors, which means cross-floor paths work similar to same-floor adjacent paths with no need for special casework.

In [4]:
# ALGORITHM: BFS (Breadth-First Search)

# HOW STAIRWELLS CREATE CROSS-FLOOR EDGES:
#   sc-stair-1 (floor 1) lists sc-stair-2 (floor 2) as a neighbor.
#   sc-stair-2 lists sc-stair-1 and sc-stair-3 as neighbors.

from collections import deque


def find_path(start_id: str, target_id: str, room_index: dict) -> list:
    """
    Find the shortest path between two rooms using Breadth-First Search.

    The graph is built step-by-step using the 'neighbors' field on each room.
    Edges are undirected and unweighted (each hop costs 1). Cross-floor
    edges exist wherever stairwell rooms list each other as neighbors.
    These are some of the key variables and ideas we have in this prototype:
    Args:
        start_id:   The room id to start from.
        target_id:  The room id to navigate to.
        room_index: {room_id: room_object} from build_room_index().

    Returns:
        A list of room ids representing the shortest path (inclusive of
        start and target), e.g. ["sc-101", "sc-102", "sc-stair-1"].
        This returns [] if no path exists (disconnected graph).

    Raises:
        ValueError: If start_id or target_id is not found in room_index.
    """
    # Test if input valid
    if start_id not in room_index:
        raise ValueError(f"Start room '{start_id}' not found in building index.")
    if target_id not in room_index:
        raise ValueError(f"Target room '{target_id}' not found in building index.")

    # TEST: start and destination are the same
    if start_id == target_id:
        return [start_id]

    queue: deque = deque([start_id])

    # not coming back to already-explored places.
    visited: set = {start_id}

    # parent maps each visited room to the room we came from
    parent: dict = {start_id: None}

    #BFS loop here
    while queue:
        current_id: str = queue.popleft()
        current_room: dict = room_index[current_id]

        for neighbor_id in current_room.get("neighbors", []):
            # Data integrity guard: skip neighbors that aren't in the index.
            if neighbor_id not in room_index:
                continue

            # already explored through a shorter or equal path, skip it
            if neighbor_id in visited:
                continue

            visited.add(neighbor_id)
            parent[neighbor_id] = current_id  #path to get here

            #stop when reaching the destination
            if neighbor_id == target_id:
                return _reconstruct_path(parent, start_id, target_id)

            queue.append(neighbor_id)


    return []


def _reconstruct_path(parent: dict, start_id: str, target_id: str) -> list:
    """
    Walk the parent dict backwards from target to start, then reverse.

    Args:
        parent:    {room_id: predecessor_id} built during BFS.
        start_id:  The origin room id.
        target_id: The destination room id.

    Returns:
        Ordered list of room ids from start to target.
    """
    path: list = []
    node = target_id
    # Follow the chain of parents until we hit the start node
    while node is not None:
        path.append(node)
        node = parent[node]
    path.reverse() 
    return path


### 5.5 Demo: End-to-End

Exercise the full pipeline: index the sample building, run two searches, and find paths for four different start/target pairs — including a same-floor hop, a cross-floor journey, and the trivial same-room case.

In [ ]:
# Helper functions for pretty output and to showcase working of above written code

def pretty_print_path(path: list, room_index: dict) -> str:
    """
    Format a path (list of room ids) as an arrow string that user can read easily.

    Each node is shown as:  id (Room Name, Floor N)

    Args:
        path:       Ordered list of room ids returned by find_path().
        room_index: {room_id: room_object} from build_room_index().

    Returns:
        A single string with nodes joined by ' -> ', or 'No path found.'
    """
    if not path:
        return "No path found."

    segments: list = []
    for room_id in path:
        room = room_index[room_id]
        segments.append(f"{room_id} ({room['name']}, Floor {room['floor']})")

    return " -> ".join(segments)


def print_search_results(results: list, query: str) -> None:
    """Pretty-print a list of rooms returned by search_rooms()."""
    print(f"\n=== Search: '{query}' - {len(results)} result(s) ===")
    if not results:
        print("  (no matches)")
        return
    for room in results:
        subtype = f" [{room['amenitySubtype']}]" if room.get("amenitySubtype") else ""
        desc = room.get("description", "")
        print(f"  [F{room['floor']}] {room['id']}  |  {room['name']}{subtype}  |  {desc}")


room_index = build_room_index(SAMPLE_BUILDING)
type_index = build_type_index(SAMPLE_BUILDING)

# search for vending machines.
# 'vending' does not appear in any room's 'type' field (they say 'amenity'),
#  but search_rooms() checks type_index keys as a second pass, catching them.
print_search_results(search_rooms("vending", room_index, type_index), "vending")

# search for offices.
print_search_results(search_rooms("office", room_index, type_index), "office")

# Path finder

# TEST 1: Floor 1 lecture hall -> Floor 3 vending machine
# expected output:
# sc-101 -> sc-102 -> sc-103 -> sc-rest-1 -> sc-stair-1
# -> sc-stair-2 -> sc-stair-3 -> sc-301 -> sc-302 -> sc-vend-1
print("\n=== Path: Lecture Hall A (F1) -> Vending Machine Alcove (F3) ===")
path_a = find_path("sc-101", "sc-vend-1", room_index)
print(pretty_print_path(path_a, room_index))
print(f"  Hops: {len(path_a) - 1}")

# TEST 2:: Cross-floor - Floor 2 computer lab -> Floor 3 vending machine
# expected output:
# sc-202 -> sc-201 -> sc-stair-2 -> sc-stair-3 -> sc-301 -> sc-302 -> sc-vend-1
print("\n=== Path: Computer Lab (F2) -> Vending Machine Alcove (F3) ===")
path_b = find_path("sc-202", "sc-vend-1", room_index)
print(pretty_print_path(path_b, room_index))
print(f"  Hops: {len(path_b) - 1}")

# TEST 3: start and destination are identical
print("\n=== Path: Lecture Hall A -> Lecture Hall A (same room) ===")
path_c = find_path("sc-101", "sc-101", room_index)
print(pretty_print_path(path_c, room_index))

# TEST 4: same floor start and destination
print("\n=== Path: Lecture Hall A (F1) -> Classroom B (F1) ===")
path_d = find_path("sc-101", "sc-102", room_index)
print(pretty_print_path(path_d, room_index))
print(f"  Hops: {len(path_d) - 1}")



=== Search: 'vending' - 1 result(s) ===
  [F3] sc-vend-1  |  Vending Machine Alcove [vending]  |  Snack and drink vending machines

=== Search: 'office' - 1 result(s) ===
  [F2] sc-201  |  Research Office  |  Faculty research office

=== Path: Lecture Hall A (F1) -> Vending Machine Alcove (F3) ===
sc-101 (Lecture Hall A, Floor 1) -> sc-102 (Classroom B, Floor 1) -> sc-103 (Lobby, Floor 1) -> sc-rest-1 (Restroom 1, Floor 1) -> sc-stair-1 (Main Stairwell (Floor 1), Floor 1) -> sc-stair-2 (Main Stairwell (Floor 2), Floor 2) -> sc-stair-3 (Main Stairwell (Floor 3), Floor 3) -> sc-301 (Study Room 3A, Floor 3) -> sc-302 (Seminar Room, Floor 3) -> sc-vend-1 (Vending Machine Alcove, Floor 3)
  Hops: 9

=== Path: Computer Lab (F2) -> Vending Machine Alcove (F3) ===
sc-202 (Computer Lab, Floor 2) -> sc-201 (Research Office, Floor 2) -> sc-stair-2 (Main Stairwell (Floor 2), Floor 2) -> sc-stair-3 (Main Stairwell (Floor 3), Floor 3) -> sc-301 (Study Room 3A, Floor 3) -> sc-302 (Seminar Room, Floo

## 6. Video Script

**Video walkthrough (5 min):**

| Time | Content | Speaker |
|---|---|---|
| 0:00 – 0:30 | Introduction | Leon |
| 0:30 – 2:30 | Walk through `SAMPLE_BUILDING` and the search function output | Leon |
| 2:30 – 05:00| Walk through the BFS algorithm and its output cases | Arjun |